# EDA ACÚSTICO — SISTEMA DE ALERTA PARA PERSONAS SORDAS

In [ ]:
from src.utils.config import EDA_BALANCED, RAW_DIR

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import umap

from tqdm import tqdm
import torch.nn as nn
from matplotlib.animation import FuncAnimation
from sklearn.utils import resample

In [ ]:
audio_eda = pd.read_csv(EDA_BALANCED)

In [ ]:
sample_path = audio_eda[audio_eda["file_exists"] == True]["path"].iloc[0]
sample_path = RAW_DIR / sample_path
y, sr = librosa.load(sample_path, sr=None)

In [ ]:
plt.figure(figsize=(12,4))

librosa.display.waveshow(y, sr=sr)

plt.title("Forma de onda del audio")
plt.show()

La señal presenta variaciones significativas de amplitud a lo largo del tiempo, evidenciando eventos acústicos no estacionarios. La presencia de múltiples picos energéticos sugiere sonidos impulsivos o de alerta, característicos de eventos relevantes para clasificación alertable.

In [ ]:
S = librosa.stft(y)
S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

plt.figure(figsize=(12,6))

librosa.display.specshow(
    S_db,
    sr=sr,
    x_axis="time",
    y_axis="hz",
    cmap="Pastel2"
)

plt.colorbar()
plt.title("Espectrograma del sonido")
plt.show()

El espectrograma evidencia actividad energética distribuida en múltiples bandas frecuenciales con transiciones temporales abruptas. Estas características indican eventos acústicos dinámicos y no estacionarios, típicos de sonidos alertables.

In [ ]:
mel = librosa.feature.melspectrogram(y=y, sr=sr)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(12,6))

librosa.display.specshow(
    mel_db,
    sr=sr,
    x_axis="time",
    y_axis="mel",
    cmap="Pastel2"
)

plt.colorbar()
plt.title("Mel Spectrogram")
plt.show()

El Mel Spectrogram presenta concentraciones energéticas predominantes en bandas medias, alineadas con la percepción auditiva humana. La variabilidad temporal confirma la presencia de eventos acústicos relevantes para tareas de detección automática de alertas.

In [ ]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

plt.figure(figsize=(12,5))

librosa.display.specshow(
    mfcc,
    x_axis="time",
    cmap="Pastel2"
)

plt.colorbar()
plt.title("MFCC del audio")
plt.show()

En la gráfica se observan patrones y cambios constantes en las características del audio a lo largo del tiempo. Algunas zonas presentan mayor intensidad y variación, lo que indica que el sonido no es uniforme y contiene diferentes componentes acústicos.

In [ ]:
def plot_mel_from_label(df, emergency_value):

    sample = df[
        (df["emergency"] == emergency_value) &
        (df["file_exists"] == True)
    ]["path"].iloc[0]
    sample = RAW_DIR / sample
    y, sr = librosa.load(sample, sr=None)

    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    plt.figure(figsize=(10,5))

    librosa.display.specshow(
        mel_db,
        sr=sr,
        x_axis="time",
        y_axis="mel",
        cmap="Pastel2"
    )

    plt.title(f"Mel Spectrogram - Emergency={emergency_value}")
    plt.colorbar()
    plt.show()

In [ ]:
plot_mel_from_label(audio_eda, 1)
plot_mel_from_label(audio_eda, 0)

### Emergency = 1

Se observa una gran cantidad de energía distribuida en distintas frecuencias y variaciones continuas durante todo el audio. Esto refleja un sonido intenso y dinámico, con presencia de frecuencias medias y altas.

### Emergency = 0

La energía aparece más concentrada y con patrones más estables. Las frecuencias altas tienen menor presencia y el audio muestra menos cambios bruscos, indicando un sonido más uniforme.

In [ ]:
zcr = librosa.feature.zero_crossing_rate(y)

plt.figure(figsize=(10,4))
plt.plot(zcr[0])
plt.title("Zero Crossing Rate")
plt.show()

La gráfica presenta varios picos pronunciados y cambios frecuentes en la señal. Esto indica que el audio tiene momentos de alta variación y sonidos más abruptos o ruidosos.

$$
ZCR = \frac{1}{2N}\sum_{n=1}^{N}|\operatorname{sign}(x_n)-\operatorname{sign}(x_{n-1})|
$$

In [ ]:
centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

plt.figure(figsize=(10,4))
plt.plot(centroid[0])
plt.title("Spectral Centroid")
plt.show()

Se observan varios aumentos importantes en la frecuencia dominante del audio. Esto significa que en ciertos momentos predominan sonidos más agudos y brillantes dentro de la señal.

$$
C = \frac{\sum_k S(k)f(k)}{\sum_k S(k)}
$$

In [ ]:
def plot_audio_analysis(audio_path, title="Audio"):

    y, sr = librosa.load(audio_path, sr=None)

    fig, ax = plt.subplots(3, 2, figsize=(16,12))
    fig.suptitle(title, fontsize=16)

    # ======================
    # 1️⃣ Waveform
    # ======================
    librosa.display.waveshow(y, sr=sr, ax=ax[0,0])
    ax[0,0].set_title("Forma de onda")

    # ======================
    # 2️⃣ Spectrogram
    # ======================
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax[0,1])
    ax[0,1].set_title("Espectrograma")
    fig.colorbar(img, ax=ax[0,1])

    # ======================
    # 3️⃣ Mel Spectrogram
    # ======================
    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    img2 = librosa.display.specshow(
        mel_db,
        sr=sr,
        x_axis="time",
        y_axis="mel",
        ax=ax[1,0]
    )
    ax[1,0].set_title("Mel Spectrogram")
    fig.colorbar(img2, ax=ax[1,0])

    # ======================
    # 4️⃣ MFCC
    # ======================
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    img3 = librosa.display.specshow(
        mfcc,
        x_axis="time",
        ax=ax[1,1]
    )
    ax[1,1].set_title("MFCC")
    fig.colorbar(img3, ax=ax[1,1])

    # ======================
    # 5️⃣ ZCR
    # ======================
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    ax[2,0].plot(zcr)
    ax[2,0].set_title("Zero Crossing Rate")

    # ======================
    # 6️⃣ Spectral Centroid
    # ======================
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    ax[2,1].plot(centroid)
    ax[2,1].set_title("Spectral Centroid")

    plt.tight_layout()
    plt.show()

### EDA acústico por dataset

In [ ]:
def sample_by_dataset(df, n=2):

    samples = []

    for dataset in df["dataset_source"].unique():

        subset = df[df["dataset_source"] == dataset]

        sampled = subset.sample(
            min(n, len(subset)),
            random_state=42
        )

        samples.append(sampled)

    return pd.concat(samples)

In [ ]:
dataset_samples = sample_by_dataset(audio_eda, n=2)

In [ ]:
for _, row in dataset_samples.iterrows():

    plot_audio_analysis(
        RAW_DIR / row["path"],
        title=f"Dataset: {row['dataset_source']} | Label: {row['human_label']}"
    )

Se observan diferencias acústicas entre datasets, evidenciando variaciones en energía, distribución espectral y complejidad temporal, lo cual justifica la combinación multifuente para mejorar la generalización del modelo.

### EDA acústico por label

In [ ]:
def sample_by_human_label(df, n=1):

    samples = []

    for label in df["human_label"].unique():

        subset = df[df["human_label"] == label]

        sampled = subset.sample(
            min(n, len(subset)),
            random_state=42
        )

        samples.append(sampled)

    return pd.concat(samples)

In [ ]:
label_samples = sample_by_human_label(audio_eda, n=1)

In [ ]:
for _, row in label_samples.iterrows():

    plot_audio_analysis(
        RAW_DIR / row["path"],
        title=f"Clase: {row['human_label']} | Dataset: {row['dataset_source']}"
    )

### Conclusión esperada

Las clases alertables presentan mayor energía transitoria, mayor variabilidad espectral y patrones acústicos distintivos respecto a clases no alertables.

In [ ]:
alert_samples = (
    audio_eda.groupby("alertable")
      .sample(2, random_state=42)
)

In [ ]:
for _, row in alert_samples.iterrows():

    plot_audio_analysis(
        RAW_DIR / row["path"],
        title=f"Alertable: {row['alertable']} | {row['human_label']}"
    )

## MAPA ACÚSTICO 2D (t-SNE / UMAP)

In [ ]:
def extract_audio_features(path):

    try:
        y, sr = librosa.load(path, sr=22050)

        features = {}

        # ZCR
        features["zcr"] = np.mean(librosa.feature.zero_crossing_rate(y))

        # Spectral features
        features["centroid"] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        features["bandwidth"] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
        features["rolloff"] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

        # RMS Energy
        features["rms"] = np.mean(librosa.feature.rms(y=y))

        # MFCC (13 coeficientes)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        for i in range(13):
            features[f"mfcc_{i}"] = np.mean(mfcc[i])

        return features

    except:
        return None

In [ ]:
feature_rows = []

for _, row in audio_eda.iterrows():

    feats = extract_audio_features(row["path"])

    if feats is not None:
        feats["human_label"] = row["human_label"]
        feats["alertable"] = row["alertable"]
        feats["dataset"] = row["dataset_source"]

        feature_rows.append(feats)

audio_features_df = pd.DataFrame(feature_rows)

In [ ]:
X = audio_features_df.drop(
    columns=["human_label","alertable","dataset"]
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### MAPA ACÚSTICO — t-SNE

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    random_state=42
)

X_tsne = tsne.fit_transform(X_scaled)

audio_features_df["tsne_1"] = X_tsne[:,0]
audio_features_df["tsne_2"] = X_tsne[:,1]

In [ ]:
plt.figure(figsize=(12,8))

palette = sns.color_palette("pastel",
            n_colors=audio_features_df["human_label"].nunique())

sns.scatterplot(
    data=audio_features_df,
    x="tsne_1",
    y="tsne_2",
    hue="human_label",
    palette=palette,
    alpha=0.8
)

plt.title("Mapa Acústico t-SNE por Clase")
plt.show()

### MAPA ACÚSTICO — UMAP 

In [ ]:
reducer = umap.UMAP(
    n_neighbors=20,
    min_dist=0.1,
    random_state=42
)

X_umap = reducer.fit_transform(X_scaled)

audio_features_df["umap_1"] = X_umap[:,0]
audio_features_df["umap_2"] = X_umap[:,1]

In [ ]:
plt.figure(figsize=(12,8))

sns.scatterplot(
    data=audio_features_df,
    x="umap_1",
    y="umap_2",
    hue="human_label",
    palette="pastel",
    alpha=0.8
)

plt.title("Mapa Acústico UMAP por Clase")
plt.show()

In [ ]:
plt.figure(figsize=(10,7))

sns.scatterplot(
    data=audio_features_df,
    x="umap_1",
    y="umap_2",
    hue="alertable",
    palette="pastel",
    s=60
)

plt.title("Separación acústica Alertable vs No Alertable")
plt.show()

La reducción dimensional mediante UMAP y t-SNE evidencia la existencia de estructuras acústicas intrínsecas dentro del espacio de características. Las clases presentan agrupamientos coherentes, lo que confirma separabilidad acústica previa al entrenamiento del modelo.

## Mapa Acústico Animado

- cómo se reorganizan los sonidos en el espacio acústico
- cómo el modelo aprende a separar clases
- evolución por épocas (epochs)

In [ ]:
def extract_features(path):

    try:
        y, sr = librosa.load(path, sr=None)

        feats = {}

        # MFCC
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        feats.update({f"mfcc_{i}": mfcc[i].mean() for i in range(13)})

        # ZCR
        feats["zcr"] = librosa.feature.zero_crossing_rate(y).mean()

        # RMS
        feats["rms"] = librosa.feature.rms(y=y).mean()

        # Spectral centroid
        feats["centroid"] = librosa.feature.spectral_centroid(y=y, sr=sr).mean()

        # Bandwidth
        feats["bandwidth"] = librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()

        # Rolloff
        feats["rolloff"] = librosa.feature.spectral_rolloff(y=y, sr=sr).mean()

        return feats

    except:
        return None

In [ ]:
audio_eda = eda[eda["file_exists"] == True].copy()

rows = []

for _, row in tqdm(audio_eda.iterrows(), total=len(audio_eda)):

    feats = extract_features(row["path"])

    if feats:
        feats["human_label"] = row["human_label"]
        feats["dataset_source"] = row["dataset_source"]

        rows.append(feats)

acoustic_df = pd.DataFrame(rows)

In [ ]:
X = acoustic_df.drop(
    columns=["human_label","dataset_source"]
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
reducer = umap.UMAP(
    n_neighbors=30,
    min_dist=0.15,
    random_state=42
)

embedding = reducer.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(10,8))

sns.scatterplot(
    x=embedding[:,0],
    y=embedding[:,1],
    hue=acoustic_df["human_label"],
    palette="pastel",
    alpha=0.7
)

plt.title("Mapa Acústico Global (Pre-Modelo)")
plt.show()

In [ ]:
progressive_embeddings = []

for n_features in range(3, X_scaled.shape[1]):

    emb = reducer.fit_transform(X_scaled[:, :n_features])
    progressive_embeddings.append(emb)

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

palette = sns.color_palette(
    "pastel",
    acoustic_df["human_label"].nunique()
)

labels = acoustic_df["human_label"].astype("category").cat.codes

def update(frame):

    ax.clear()

    emb = progressive_embeddings[frame]

    ax.scatter(
        emb[:,0],
        emb[:,1],
        c=[palette[l] for l in labels],
        alpha=0.7
    )

    ax.set_title(
        f"Estructura acústica emergente — Features usadas: {frame+3}"
    )

anim = FuncAnimation(
    fig,
    update,
    frames=len(progressive_embeddings),
    interval=700
)

plt.show()

In [ ]:
anim.save(
    "acoustic_learning.gif",
    writer="pillow",
    fps=2
)